In [ ]:
# import packages
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import RendererAgg
from matplotlib.markers import MarkerStyle
from matplotlib.font_manager import FontProperties
from PIL import Image

In [ ]:
# settings
tracker_path = 'tracker.csv'
icon_path = 'images/traitorsIcon.png'

# the chart is for the latest episode with anything in the tracker; set a number to re-render an earlier week
current_episode = None

# annotations: (text, player name, episode the arrow points at, box top-left x px, box top-left y px)
# the episode is a number, or e.g. '3_II' to point at one round table of an episode that had two
# e.g. ('Rob has spotted a traitor\nevery single week', 'Rob Beckett', 5, 720, 470)
annotations = []

In [ ]:
# setup data
# tracker.csv has one row per player; update it after each episode:
#   Traitor_From_Episode  blank for faithfuls, else the episode they became a traitor (1 for original traitors)
#   Eliminated_Episode    blank while they're still in, else the episode they were banished or murdered; for an
#                         episode with two round tables say which, e.g. 3_II (a bare 3 means the last one)
#   Ep1, Ep2, ...         one column per episode in the series: Y if they voted for a traitor at that round table,
#                         N if not; leave the whole column blank for an episode with no round table vote.
#                         An episode with two round tables has a column for each instead, e.g. Ep3_I and Ep3_II
players = pd.read_csv(tracker_path, dtype={'Eliminated_Episode': 'string'})
players['Name'] = players['Name'].str.strip()

# one chart column ("slot") per round table, in order; an episode with several round tables numbers them
romans = ['I', 'II', 'III', 'IV', 'V', 'VI', 'VII', 'VIII', 'IX', 'X']
found = {}
for c in players.columns:
    match = re.fullmatch(r'Ep(\d+)(?:_(' + '|'.join(romans) + '))?', c)
    if match:
        found.setdefault(int(match.group(1)), {})[match.group(2) or ''] = c
n_episodes = max(found)
slots = []
for ep in range(1, n_episodes + 1):
    parts = found.get(ep, {'': f'Ep{ep}'})  # an episode with no column in the tracker yet is treated as blank
    for part_index, part in enumerate(sorted(parts, key=lambda p: romans.index(p) if p else -1)):
        slots.append(dict(episode=ep, part=part_index, col=parts[part],
                          roman=romans[part_index] if len(parts) > 1 else ''))
slot_cols = [slot['col'] for slot in slots]
for c in slot_cols:
    players[c] = players[c].astype('string') if c in players.columns else pd.Series(pd.NA, index=players.index, dtype='string')

answers = {'y': True, 'yes': True, 'n': False, 'no': False}
votes = players[slot_cols].apply(lambda col: col.str.strip().str.lower().map(answers))

problems = []

def find_slot(spec, which='last'):
    """The chart column for an episode ('3' or 3) or one of its round tables ('3_II'); None if there isn't one."""
    ep_text, _, roman = str(spec).strip().upper().partition('_')
    options = [i for i, slot in enumerate(slots) if slot['episode'] == int(float(ep_text))]
    if roman:
        options = [i for i in options if slots[i]['roman'] == roman]
    if not options:
        return None
    return options[-1] if which == 'last' else options[0]

# an elimination is an episode, optionally with its round table (3 or 3_II); keep both the episode and the column
players['Eliminated_Slot'] = np.nan
for i, spec in players['Eliminated_Episode'].items():
    if pd.isna(spec):
        continue
    slot = find_slot(spec)
    if slot is None:
        problems.append(f"{players.at[i, 'Name']} Eliminated_Episode: '{spec}' doesn't match a round table (ignored)")
    else:
        players.at[i, 'Eliminated_Slot'] = slot
players['Eliminated_Episode'] = players['Eliminated_Slot'].map(lambda s: slots[int(s)]['episode'] if pd.notna(s) else np.nan)

if current_episode is None:
    entered = ([slot['episode'] for slot in slots if players[slot['col']].notna().any()]
               + players['Traitor_From_Episode'].dropna().tolist()
               + players['Eliminated_Episode'].dropna().tolist())
    current_episode = int(max(entered, default=1))
output_path = f'images/traitors_ep{current_episode}.png'

# eliminations and recruitments after the current episode haven't aired, so they're ignored
players['Out'] = players['Eliminated_Episode'] <= current_episode
players['Traitor'] = players['Traitor_From_Episode'] <= current_episode

# round tables still to come: a later one in the current episode counts as played once it has a vote or an elimination
voting_slots = {i for i, slot in enumerate(slots) if players[slot['col']].notna().any()}
played_slots = voting_slots | {int(s) for s in players.loc[players['Out'], 'Eliminated_Slot']}
slot_future = [slot['episode'] > current_episode
               or (slot['episode'] == current_episode and slot['part'] > 0 and i not in played_slots)
               for i, slot in enumerate(slots)]
current_slot = max((i for i, future in enumerate(slot_future) if not future), default=-1)

# check the tracker: anything unreadable, votes after a player was eliminated, and any vote missing
# for a round table the player was in (a column that is entirely blank had no round table vote)
for i, player in players.iterrows():
    last_vote = player['Eliminated_Slot'] - 1 if player['Out'] else current_slot
    for k, slot in enumerate(slots):
        raw = player[slot['col']]
        if pd.notna(raw) and pd.isna(votes.at[i, slot['col']]):
            problems.append(f"{player['Name']} {slot['col']}: '{raw}' isn't Y or N")
        elif pd.notna(raw) and k > player['Eliminated_Slot']:
            problems.append(f"{player['Name']} {slot['col']}: vote entered after they were eliminated in episode "
                            f"{player['Eliminated_Episode']:.0f} (ignored)")
        elif k <= last_vote and k in voting_slots and pd.isna(raw):
            problems.append(f"{player['Name']} {slot['col']}: no vote entered (drawn as not voting for a traitor)")
if problems:
    warnings.warn('Check tracker.csv:\n' + '\n'.join(problems))

players[slot_cols] = votes.eq(True)

def cell_status(row, slot):
    if row['Out'] and slot > row['Eliminated_Slot']:
        return None
    if row['Out'] and slot == row['Eliminated_Slot']:
        return 'Eliminated'
    if slot_future[slot]:
        return 'Future'
    return 'Traitor vote' if row[slots[slot]['col']] else 'Faithful vote'

# display names
players['Label'] = (players['Name']
                    .str.replace('Professor ', 'Prof. ', regex=False)
                    .str.replace("'", '’', regex=False))

In [ ]:
# order: players still in by surname, then eliminated players, most recent first
players['Surname'] = players['Name'].str.split().str[-1]
still_in = players[~players['Out']].sort_values('Surname')
out = players[players['Out']].sort_values(['Eliminated_Slot', 'Traitor', 'Surname'],
                                          ascending=[False, False, True])
players = pd.concat([still_in, out]).reset_index(drop=True)

In [ ]:
# style
# every measurement is in pixels on the 1080x1080 canvas
dpi = 300
width_px, height_px = 1080, 1080
margin_px = 50
# pull the grid of dots in this far from the right and bottom margins
right_inset_px = 20
bottom_inset_px = 45  # leaves room above the note

def pt(px):
    """Convert canvas pixels to matplotlib points."""
    return px * 72 / dpi

background = '#F7F1E1'
ink = '#1E1F1E'
orange = '#FF8F01'
sand = '#E2C58B'
sand_light = '#EBD6AE'
maroon = '#7D0038'
annotation_stroke = '#A44D74'
grey = '#8F8F8F'
out_alpha = 0.4  # eliminated players are drawn faded

font_size = pt(22)
# numeric weights: matplotlib's 'light' means 200, which would pick Poppins Thin
light = FontProperties(family='Poppins', weight=300, size=font_size)
regular = FontProperties(family='Poppins', weight=400, size=font_size)
semibold = FontProperties(family='Poppins', weight=600, size=font_size)
cap_offset = 0.35 * 22  # baseline sits this far below a row's centre, centring cap height
# roman numeral subscript beside an episode number: font size, drop below the number's baseline, gap
sub_size, sub_drop, sub_gap = 15, 4.5, 1

# note along the bottom margin, below the grid
note = 'Note: Traitors logo/branding are properties of IDTV, Studio Lambert and the BBC.'
note_font = FontProperties(family='Poppins', weight=300, size=pt(14))

measure = RendererAgg(width_px, height_px, dpi)

def text_width(s, prop):
    """Width of a piece of text in canvas pixels."""
    return measure.get_text_width_height_descent(s, prop, ismath=False)[0]

# marks
icon_px = 35
dot = dict(traitor=28, small=8)
x_marker = MarkerStyle('x', capstyle='round')
x_size, x_width = 18, 6

# layout
name_x = margin_px + icon_px + 8
legend_y = margin_px + icon_px / 2
legend_gap = 40  # space between one legend item and the next
header_y = 136

# round table columns run from col_first_x out to the grid's right edge, however many there are;
# a vote dot in the last column just touches that edge
grid_right = width_px - margin_px - right_inset_px
grid_bottom = height_px - margin_px - bottom_inset_px
col_first_x = 408
col_last_x = grid_right - dot['traitor'] / 2
col_pitch = (col_last_x - col_first_x) / max(len(slots) - 1, 1)

# player rows run from row_y0 down to the grid's bottom edge; the tallest thing in the bottom row
# is a vote dot, or the traitor icon if that player is a traitor
row_y0 = 180
bottom_half = max(dot['traitor'], icon_px if players['Traitor'].iloc[-1] else 0) / 2
row_pitch = (grid_bottom - bottom_half - row_y0) / max(len(players) - 1, 1)

def col_x(slot):
    return col_first_x + slot * col_pitch

def row_y(i):
    return row_y0 + i * row_pitch

# icon, cropped to the logo and resized to the exact pixel size
icon = Image.open(icon_path).convert('RGBA')
icon = icon.crop(icon.getchannel('A').getbbox())
icon_w = round(icon_px * icon.width / icon.height)
icon = np.asarray(icon.resize((icon_w, icon_px), Image.LANCZOS))

In [ ]:
# create plot: one axes over the whole canvas, in pixel coordinates with y pointing down
fig = plt.figure(figsize=(width_px / dpi, height_px / dpi), dpi=dpi, facecolor=background)
ax = fig.add_axes([0, 0, 1, 1])
ax.set_xlim(0, width_px)
ax.set_ylim(height_px, 0)
ax.axis('off')

def draw_icon(x, y_centre, alpha=1):
    y0 = round(y_centre - icon_px / 2)
    ax.imshow(icon, extent=[x, x + icon_w, y0 + icon_px, y0], alpha=alpha,
              interpolation='nearest', aspect='auto', zorder=3)

def draw_dot(x, y, kind, alpha=1):
    if kind == 'Traitor vote':
        ax.scatter(x, y, s=pt(dot['traitor']) ** 2, c=orange, edgecolors=ink, linewidths=pt(1), alpha=alpha, zorder=3)
    elif kind == 'Faithful vote':
        ax.scatter(x, y, s=pt(dot['small']) ** 2, c=sand, linewidths=0, alpha=alpha, zorder=3)
    elif kind == 'Future':
        ax.scatter(x, y, s=pt(dot['small']) ** 2, c=background, edgecolors=sand_light, linewidths=pt(1), zorder=3)
    elif kind == 'Eliminated':
        ax.plot(x, y, marker=x_marker, markersize=pt(x_size), markeredgewidth=pt(x_width),
                color=maroon, alpha=alpha, linestyle='none', zorder=3)

# legend
x = margin_px
legend = [('icon', 'Traitor'), ('Traitor vote', 'Voted for a traitor'), ('Eliminated', 'Eliminated')]
for marker, label in legend:
    if marker == 'icon':
        draw_icon(x, legend_y)
        x += icon_w
    else:
        size = dot['traitor'] if marker == 'Traitor vote' else x_size + x_width
        if marker == 'Eliminated':
            # centred above the episode 2 column, as in the design, unless that would crowd the label before it
            x = max(x - legend_gap / 2, col_x(1) - size / 2)
        draw_dot(x + size / 2, legend_y, marker)
        x += size
    x += 10
    ax.text(x, legend_y + cap_offset, label, fontproperties=light, color=ink, va='baseline')
    x += text_width(label, light) + legend_gap

# column headers, centred on their column; a header wider than the space left before the right margin
# (the finale's bold 'Ep N') is nudged left to stay inside it. An episode with two round tables has a
# column for each, with its roman numeral as a subscript beside the number
for k, slot in enumerate(slots):
    ep = slot['episode']
    if ep == current_episode:
        label, prop, colour = (f'Ep {ep}' if slot['part'] == 0 else str(ep)), semibold, ink
    else:
        label, prop, colour = str(ep), light, ink if ep < current_episode else sand
    sub_prop = FontProperties(family='Poppins', weight=prop.get_weight(), size=pt(sub_size))
    sub_space = sub_gap + text_width(slot['roman'], sub_prop) if slot['roman'] else 0
    x = min(col_x(k), width_px - margin_px - text_width(label, prop) / 2 - sub_space)
    ax.text(x, header_y + cap_offset, label, fontproperties=prop, color=colour, ha='center', va='baseline')
    if slot['roman']:
        ax.text(x + text_width(label, prop) / 2 + sub_gap, header_y + cap_offset + sub_drop, slot['roman'],
                fontproperties=sub_prop, color=colour, ha='left', va='baseline')

# players: traitors get the icon and a bold name, eliminated players are faded
for i, player in players.iterrows():
    y = row_y(i)
    alpha = out_alpha if player['Out'] else 1
    if player['Traitor']:
        draw_icon(margin_px, y, alpha)
    prop = semibold if player['Traitor'] else light
    ax.text(name_x, y + cap_offset, player['Label'], fontproperties=prop, color=ink, alpha=alpha, va='baseline')

    for k, slot in enumerate(slots):
        status = cell_status(player, k)
        if status is None:
            continue
        # this week's elimination stays at full strength
        cell_alpha = 1 if status == 'Eliminated' and slot['episode'] == current_episode else alpha
        draw_dot(col_x(k), y, status, cell_alpha)

# note, lined up with the names, its bottom on the bottom margin
ax.text(name_x, height_px - margin_px, note, fontproperties=note_font, color=grey, va='bottom')

# annotations
# matplotlib steps lines by the font's own line height x linespacing, so measure it to get 30px
renderer = fig.canvas.get_renderer()
one_line, two_lines = (ax.text(0, 0, s, fontproperties=regular, linespacing=1) for s in ('A', 'A\nA'))
font_line_px = (two_lines.get_window_extent(renderer).height
                - one_line.get_window_extent(renderer).height) * width_px / fig.bbox.width
one_line.remove(), two_lines.remove()
line_spacing = 30 / font_line_px

row_of = {name: i for i, name in enumerate(players['Name'])}
for text, name, ep, box_x, box_y in annotations:
    pad = 12
    slot = find_slot(ep, 'first')
    if slot is None:
        raise ValueError(f'annotation for {name} points at episode {ep}, which has no column')
    ax.annotate(text, xy=(col_x(slot), row_y(row_of[name])), xytext=(box_x + pad, box_y + pad),
                fontproperties=regular, color=maroon, linespacing=line_spacing, ha='left', va='top',
                bbox=dict(boxstyle=f'square,pad={pad / 22}', facecolor='#FFFFFF',
                          edgecolor=annotation_stroke, linewidth=pt(1.5)),
                arrowprops=dict(arrowstyle='-', color=annotation_stroke, linewidth=pt(1.5),
                                shrinkA=0, shrinkB=pt(dot['traitor'] / 2 + 4)),
                zorder=4)

# save final plot
fig.savefig(output_path, dpi=dpi, facecolor=background)
plt.show()